# 21 · Aggregate points to H3 cells, checked against h3-py

## What this checks

**Aggregate points to H3 cells** counts points in hexagonal cells, instead of showing each point. H3 is a worldwide
grid of hexagons, with cells at many sizes called resolutions. This check covers release 0.5.0.

The widget has a rule of its own for sparse cells. A cell with fewer points than a stated minimum is **left out
entirely**. It is not shown with a small count, so its points disappear from the output. The widget's note says that
this minimum is not a guarantee of anonymity. What it does guarantee is that no cell shown holds fewer than the
minimum. The price is that some points are not represented at all.

## How it is checked

Fieldwork places points in cells with h3-js. The reference uses **h3-py**. These are two links, one for JavaScript
and one for Python, to the same H3 library. So the placing in cells is **not** checked independently here, and this
check does not pretend otherwise. What it checks is that Fieldwork calls H3 correctly, orders the result as it says,
and applies its own sparse-cell rule.

The data are 250 deaths from the John Snow example, with a minimum of 2 points a cell. There are two cases, because
the example's own setting does not test the rule it depends on:

- **Resolution 9.** All 250 deaths fall into five cells, all above the minimum, so nothing is dropped.
- **Resolution 11.** This gives cells with a single point, the case that makes points vanish.

## What should happen

For each case:

1. **The cells kept** are the same, exactly.
2. **The count in each cell** is the same, exactly.
3. **Each cell's boundary and centre** match h3-py's within 10⁻⁹ degrees.
4. **The order** follows the H3 index, as the widget states.
5. **The accounting** matches exactly: how many points were placed, and how many the sparse rule dropped.

These criteria were fixed before the check was run.

In [ ]:
# Only what this notebook imports. Requesting rasterio alongside pyproj kills this kernel
# outright; see 00-kernel-check for the measured map.
NEEDED = ["h3"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_21_hex import run

report = run(Path.cwd() / "fixtures" / "hex-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"], "over", report["points"], "points, minimum occupancy",
      report["minOccupancy"])
print()
print(f"{'resolution':>11} {'cells kept':>11} {'cells dropped':>14} {'points removed':>15} {'worst coord diff':>18}")
for case in report["cases"]:
    worst = max(case["worstBoundaryDifferenceDegrees"], case["worstCentreDifferenceDegrees"])
    print(f"{case['resolution']:>11} {case['cellsKeptReference']:>11} "
          f"{case['cellsDroppedBySparseRule']:>14} {case['pointsDroppedBySparseRule']:>15} "
          f"{worst:>18.1e}")
print()
print(report["environment"]["note"])

## The result on record

First recorded on 11 October 2026, against release 0.2.0. Rerun on 26 November 2026, against release 0.5.0:
**Agrees**.

- **Resolution 9.** Both sides kept 5 cells, and nothing was dropped.
- **Resolution 11.** Both sides kept 52 cells. The sparse rule withheld 13 cells, and so removed 13 of the 250 deaths
  from the output.

In both cases the cells, counts and order matched exactly. Every boundary and centre was within about
3.6 × 10⁻¹⁴ degrees.

Running the cell above repeats the check on the same data.

## What this does not show

- That H3 places points in the right cells. Both sides call the same C library through different links, so agreement
  is expected. It says nothing about the index itself.
- That grouping in cells protects anybody. The minimum is not a guarantee of anonymity.
- That every point is shown. At resolution 11 the rule silently removes 13 of 250 deaths from the output, instead of
  saying so.